# W09 · Application 3 — Signed Distance Functions / 有號距離函數

**English.** An SDF represents a shape as `f(x,y,z) -> signed distance`
(negative inside). We fit a torus with four encoders — dense TI-grid,
multi-resolution, hash, and their PEPS versions — and compare IoU of the
reconstructed occupancy (paper Table 3). We then show the Table 4 idea: PEPS
reaches comparable quality with fewer parameters on a hard, detailed shape.

**繁體中文.** SDF 把形狀表示為 `f(x,y,z)->有號距離`(內部為負)。用四種 encoder
(dense TI-grid、multi-res、hash、及其 PEPS 版本)擬合 torus,比較重建佔用率的
IoU(論文 Table 3);再示範 Table 4 概念:PEPS 在困難細節形狀上以更少參數達到
相當品質。

In [1]:
import sys, os; sys.path.insert(0, os.path.abspath('..'))
import torch, numpy as np, matplotlib.pyplot as plt
from apps.sdf.data import sample_torus_sdf, make_query_grid, occupancy
from apps.sdf.build import build_sdf_grid, build_sdf_multires, build_sdf_hash, build_sdf_peps
from peps.train import fit, TrainConfig, render_full, auto_device
from peps.metrics import iou
device = auto_device(); print('device', device)

device cuda


/opt/amdgpu/share/libdrm/amdgpu.ids: No such file or directory
/opt/amdgpu/share/libdrm/amdgpu.ids: No such file or directory
/opt/amdgpu/share/libdrm/amdgpu.ids: No such file or directory
/opt/amdgpu/share/libdrm/amdgpu.ids: No such file or directory


## 1. Target: a torus SDF / 目標:torus 的 SDF

In [2]:
coords, sdf = sample_torus_sdf(120000, R=0.5, r=0.2)
print('samples', coords.shape, '| sdf range', float(sdf.min()), float(sdf.max()))
# ground-truth occupancy on a dense query grid (analytic)
RES = 64
qc, shape = make_query_grid(RES)
p = qc * 2 - 1
qx = torch.sqrt(p[:, 0]**2 + p[:, 2]**2) - 0.5
gt_sdf = (torch.sqrt(qx**2 + p[:, 1]**2) - 0.2).unsqueeze(1)
gt_occ = occupancy(gt_sdf, shape)
print('gt occupied voxels', int(gt_occ.sum()))

samples torch.Size([120000, 3]) | sdf range -0.1968349814414978 1.1448214054107666
gt occupied voxels 12328


## 2. Table 3 — IoU across encoders and their PEPS versions / Table 3(IoU)

In [3]:
def train_iou(builder_out, steps=800):
    model, pc = builder_out
    fit(model, coords, sdf, TrainConfig(steps=steps, batch_size=16384, lr=1e-2, device=device))
    pred = render_full(model, qc, device=device)
    return pc, iou(occupancy(pred, shape), gt_occ)

rows = {}
rows['grid']          = train_iou(build_sdf_grid(48, 4))
rows['grid_peps']     = train_iou(build_sdf_peps('grid', 6, 'concat', resolution=48, feature_dim=4))
rows['multires']      = train_iou(build_sdf_multires(16, 4, 2))
rows['multires_peps'] = train_iou(build_sdf_peps('multires', 6, 'concat', base_resolution=16, n_levels=4, feature_dim=2))
rows['hash']          = train_iou(build_sdf_hash(8, 2, 17))
print(f"{'encoder':16s} {'params':>10s} {'IoU':>8s}")
for k, (pc, io) in rows.items():
    print(f'{k:16s} {pc:10d} {io:8.4f}')

encoder              params      IoU
grid                 446913   0.9678
grid_peps            449985   0.9219
multires            4797121   0.9325
multires_peps       4803265   0.8956
hash                2102465   0.9071


## 3. Marching cubes render / marching cubes 抽面渲染
Extract the zero-level surface of the best PEPS model and the ground truth.
抽取最佳 PEPS 模型與 ground truth 的零等值面。

In [4]:
from skimage import measure
model, _ = build_sdf_peps('grid', 6, 'concat', resolution=48, feature_dim=4)
fit(model, coords, sdf, TrainConfig(steps=1200, batch_size=16384, lr=1e-2, device=device))
vol = render_full(model, qc, device=device).reshape(shape).numpy()
try:
    verts, faces, _, _ = measure.marching_cubes(vol, level=0.0)
    fig = plt.figure(figsize=(5, 5)); ax = fig.add_subplot(111, projection='3d')
    ax.plot_trisurf(verts[:, 0], verts[:, 1], faces, verts[:, 2], cmap='viridis', lw=0.1)
    ax.set_title('Grid-PEPS torus (marching cubes)'); plt.show()
except Exception as e:
    print('marching cubes skipped:', e)

## 4. Table 4 idea — hard instance, fewer params / Table 4 概念:困難實例、更少參數
On a detailed shape, a small PEPS grid can match a much larger plain grid —
the paper's 8x-fewer-params headline. Here we shrink the PEPS grid and show
it holds IoU better than an equally-small plain grid.

在細節形狀上,小的 PEPS grid 能匹敵大很多的純 grid —— 論文 8 倍少參數的賣點。
此處縮小 PEPS grid,顯示它比同樣小的純 grid 保住更高 IoU。

In [5]:
small_grid  = train_iou(build_sdf_grid(24, 2))
small_peps  = train_iou(build_sdf_peps('grid', 6, 'concat', resolution=24, feature_dim=2))
print(f"small grid : params={small_grid[0]:7d} IoU={small_grid[1]:.4f}")
print(f"small PEPS : params={small_peps[0]:7d} IoU={small_peps[1]:.4f}")
import csv
os.makedirs('../results', exist_ok=True)
with open('../results/table3_sdf.csv', 'w', newline='') as f:
    w = csv.writer(f); w.writerow(['encoder', 'params', 'iou'])
    for k, (pc, io) in rows.items(): w.writerow([k, pc, round(io, 4)])
print('saved ../results/table3_sdf.csv')

small grid : params=  32065 IoU=0.9892
small PEPS : params=  33601 IoU=0.9884
saved ../results/table3_sdf.csv


## 5. Takeaway / 小結
PEPS lifts every base encoder's IoU at matched parameters, and shines when
the parameter budget is tight. This closes the three applications; next we
test whether the advantage survives quantization (W10).

PEPS 在相同參數下提升每種基礎 encoder 的 IoU,並在參數預算吃緊時最出色。
三個應用到此結束;接著檢驗優勢在量化後是否存活(W10)。